# Notebook 11: Color Waypoint + Obstacle Challenge

## ADAS Connection
Real autonomous vehicles do not rely on a single system to make decisions. They layer
computer vision, sensor data, and AI reasoning together. In this challenge you will
see what happens when you swap those layers in and out -- and which one performs better
under pressure.

Your robot will respond to colored cards held up by the judge AND avoid obstacles it
encounters while driving. You will run the challenge twice -- once using pure computer
vision and sensor logic, once using an AI model as the decision maker.

---

## The Rules

### Color Cards
| Card Color | Action |
|------------|--------|
| Red | Stop for 2 seconds |
| Green | Go forward |
| Orange | Mission complete -- stop |

### Obstacle Avoidance
| Situation | Action |
|-----------|--------|
| Obstacle detected on the left | Turn right |
| Obstacle detected on the right | Turn left |
| Obstacle directly ahead | Stop |

- The judge holds up a color card and your robot must respond correctly
- The robot also avoids obstacles it encounters while driving
- In **CV Mode** decisions are instant -- color map + sensor logic
- In **AI Mode** the robot stops and waits while Phi-3 reasons about what to do

---

## Before You Start

> **Step 1:** Make sure your robot is connected to the school WiFi
>
> **Step 2:** Update `SERVER_IP` in the configuration cell -- your instructor will give you the address
>
> **Step 3:** Run the setup cell, then the configuration cell, then the challenge cell.

---

## Setup
Run this cell once. **Do not modify.**

In [16]:
import cv2
import numpy as np
import RPi.GPIO as GPIO
import requests
import time
import ipywidgets as widgets
from IPython.display import display

# ── Camera ────────────────────────────────────────────────────
if 'cap' not in dir() or not cap.isOpened():
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    cap.set(cv2.CAP_PROP_FPS, 30)
    cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter.fourcc('M','J','P','G'))
    for _ in range(20):
        cap.read()
    print('Camera initialized!')
else:
    print('Camera already open -- skipping init.')

# ── Motor pins (BCM numbering) ────────────────────────────────
LEFT_GO    = 20
LEFT_BACK  = 21
LEFT_PWM   = 16
RIGHT_GO   = 19
RIGHT_BACK = 6
RIGHT_PWM  = 13

# ── Ultrasonic pins (BCM numbering) ──────────────────────────
TRIG = 1
ECHO = 0

GPIO.setmode(GPIO.BCM)
GPIO.setwarnings(False)

for pin in [LEFT_GO, LEFT_BACK, LEFT_PWM, RIGHT_GO, RIGHT_BACK, RIGHT_PWM]:
    GPIO.setup(pin, GPIO.OUT)
GPIO.setup(TRIG, GPIO.OUT)
GPIO.setup(ECHO, GPIO.IN)
GPIO.output(TRIG, GPIO.LOW)

pwm_left  = GPIO.PWM(LEFT_PWM,  100)
pwm_right = GPIO.PWM(RIGHT_PWM, 100)
pwm_left.start(0)
pwm_right.start(0)

# ── Motor functions ───────────────────────────────────────────
def forward(speed=60):
    GPIO.output(LEFT_GO,    GPIO.HIGH)
    GPIO.output(LEFT_BACK,  GPIO.LOW)
    GPIO.output(RIGHT_GO,   GPIO.HIGH)
    GPIO.output(RIGHT_BACK, GPIO.LOW)
    pwm_left.ChangeDutyCycle(speed)
    pwm_right.ChangeDutyCycle(speed)

def stop():
    GPIO.output(LEFT_GO,    GPIO.LOW)
    GPIO.output(LEFT_BACK,  GPIO.LOW)
    GPIO.output(RIGHT_GO,   GPIO.LOW)
    GPIO.output(RIGHT_BACK, GPIO.LOW)
    pwm_left.ChangeDutyCycle(0)
    pwm_right.ChangeDutyCycle(0)

def turn_left(speed=60, duration=0.5):
    GPIO.output(LEFT_GO,    GPIO.LOW)
    GPIO.output(LEFT_BACK,  GPIO.HIGH)
    GPIO.output(RIGHT_GO,   GPIO.HIGH)
    GPIO.output(RIGHT_BACK, GPIO.LOW)
    pwm_left.ChangeDutyCycle(speed)
    pwm_right.ChangeDutyCycle(speed)
    time.sleep(duration)
    stop()

def turn_right(speed=60, duration=0.5):
    GPIO.output(LEFT_GO,    GPIO.HIGH)
    GPIO.output(LEFT_BACK,  GPIO.LOW)
    GPIO.output(RIGHT_GO,   GPIO.LOW)
    GPIO.output(RIGHT_BACK, GPIO.HIGH)
    pwm_left.ChangeDutyCycle(speed)
    pwm_right.ChangeDutyCycle(speed)
    time.sleep(duration)
    stop()

# ── Sensor functions ──────────────────────────────────────────
def get_distance():
    """Measure distance in cm using the ultrasonic sensor."""
    GPIO.output(TRIG, GPIO.HIGH)
    time.sleep(0.00001)
    GPIO.output(TRIG, GPIO.LOW)
    pulse_start = time.time()
    while GPIO.input(ECHO) == 0:
        pulse_start = time.time()
    pulse_end = time.time()
    while GPIO.input(ECHO) == 1:
        pulse_end = time.time()
    return round((pulse_end - pulse_start) * 34300 / 2, 1)

def get_obstacle():
    """Return 'left', 'right', 'ahead', or None based on tracking sensors and distance.
    Tracking sensors detect which side an obstacle is on.
    Ultrasonic confirms something is close enough to act on.
    """
    dist = get_distance()
    if dist > OBSTACLE_DISTANCE:
        return 1
    return 0

# ── Color ranges (HSV) ────────────────────────────────────────
COLOR_RANGES = {
    'red':    ([0,   43,  46],  [10,  255, 255]),
    'green':  ([35,  43,  46],  [77,  255, 255]),
    'orange': ([11,  43,  46],  [25,  255, 255]),
}

# ── Helper functions ──────────────────────────────────────────
def bgr8_to_jpeg(frame):
    return bytes(cv2.imencode('.jpg', frame)[1])

def detect_color(frame):
    """Return the dominant color name detected in the frame, or None."""
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    best_color = None
    best_count = 500
    for color, (lower, upper) in COLOR_RANGES.items():
        mask = cv2.inRange(hsv, np.array(lower), np.array(upper))
        count = cv2.countNonZero(mask)
        if count > best_count:
            best_count = count
            best_color = color
    return best_color

def ask_ai_sequence(situation):
    """Ask Phi-3 for a sequence of actions to handle a situation."""
    prompt = (
        f'You are the decision-making system for an autonomous robot. '
        f'The robot is trying to drive straight from point A to point B. '
        f'{situation} '
        f'Give me a sequence of actions and durations in seconds to navigate around it '
        f'and resume the original path. '
        f'Each action must be on its own line in exactly this format: ACTION:SECONDS '
        f'Valid actions are: LEFT, RIGHT, GO, STOP. '
        f'Do not include any other text, explanation, or punctuation.'
    )
    try:
        response = requests.post(
            f'http://{SERVER_IP}:{SERVER_PORT}/api/generate',
            json={'model': 'phi3:mini', 'prompt': prompt, 'stream': False},
            timeout=10
        )
        raw = response.json()['response']
        return raw, parse_ai_sequence(raw)
    except Exception as e:
        print(f'  AI error: {e}')
        return None, None
    
def execute_avoidance_sequence(sequence):
    """Execute a list of (action, duration) tuples."""
    for action, duration in sequence:
        print(f'  Executing: {action} for {duration}s')
        if action == 'LEFT':
            turn_left(DRIVE_SPEED, duration)
        elif action == 'RIGHT':
            turn_right(DRIVE_SPEED, duration)
        elif action == 'GO':
            forward(DRIVE_SPEED)
            time.sleep(duration)
        elif action == 'STOP':
            stop()
            time.sleep(duration)
    forward(DRIVE_SPEED)

def parse_ai_sequence(response):
    """Parse ACTION:SECONDS lines from AI response.
    Returns list of (action, duration) tuples, or None if unparseable.
    """
    valid_actions = ['LEFT', 'RIGHT', 'GO', 'STOP']
    sequence = []
    for line in response.strip().upper().splitlines():
        line = line.strip()
        if ':' not in line:
            continue
        parts = line.split(':')
        if len(parts) != 2:
            continue
        action = parts[0].strip()
        try:
            duration = float(parts[1].strip())
        except ValueError:
            continue
        if action in valid_actions:
            sequence.append((action, duration))
    return sequence if sequence else None

#TWEAK THIS VALUE
DRIVE_SPEED       = 20             # motor speed (0-100)

def execute_action(action, speed = DRIVE_SPEED):
    """Execute the driving action."""
    if action == 'GO':
        forward(speed)
    elif action == 'STOP':
        stop()
        time.sleep(2)
        forward(speed)
    elif action == 'LEFT':
        turn_left(speed, TURN_DURATION)
        forward(speed)
    elif action == 'RIGHT':
        turn_right(speed, TURN_DURATION)
        forward(speed)
    elif action == 'SLOW':
        speed = speed-10
        forward(speed)
    elif action == 'DONE':
        stop()
    return speed

time.sleep(0.5)
print('Setup complete!')
print('Camera, motors, and ultrasonic ready.')
print()
print('Next: run the configuration cell.')

Camera initialized!
Setup complete!
Camera, motors, and ultrasonic ready.

Next: run the configuration cell.


---

## Configuration
**Run this cell to set your mode and server address before the challenge.**

Re-run to switch between CV and AI mode between runs.

In [18]:
# ═══════════════════════════════════════
#   TWEAK THESE VALUES
MODE              = 'CV'           # 'CV' for computer vision, 'AI' for Phi-3
SERVER_IP         = '192.168.4.44' # ask your instructor for the AI server IP
SERVER_PORT       = 11434          # default Ollama port -- do not change
DRIVE_SPEED       = 20             # motor speed (0-100)
TURN_DURATION     = 0.5            # seconds per turn -- tune this for your kit
SCAN_INTERVAL     = 0.2            # seconds between camera scans
OBSTACLE_DISTANCE = 25             # cm -- react if obstacle closer than this
# ═══════════════════════════════════════

print(f'Mode:              {MODE}')
print(f'Drive speed:       {DRIVE_SPEED}')
print(f'Turn duration:     {TURN_DURATION}s')
print(f'Obstacle distance: {OBSTACLE_DISTANCE}cm')
if MODE == 'AI':
    print(f'AI server:         http://192.168.4.44:11434')
    if SERVER_IP == '192.168.4.44':
        print()
        print('  WARNING: You have not set the server IP address!')
        print('  AI Mode will not work until you update SERVER_IP.')
print()
print('Configuration ready. Run the challenge cell when the judge says go!')

Mode:              CV
Drive speed:       20
Turn duration:     0.5s
Obstacle distance: 25cm

Configuration ready. Run the challenge cell when the judge says go!


---

## Challenge
Run this cell to start. The robot will scan for color cards and avoid obstacles.
Run the **Stop** cell at any time to end the run.

In [19]:
print('=' * 50)
print(f'  CHALLENGE STARTING -- Mode: {MODE}')
print('=' * 50)
print()

challenge_running = True
last_action = None

feed_widget = widgets.Image(format='jpeg', width=640, height=480)
display(feed_widget)

#TWEAK THIS VALUE
DRIVE_SPEED       = 20             # motor speed (0-100)

def execute_action(action, speed = DRIVE_SPEED):
    """Execute the driving action."""
    if action == 'GO':
        forward(speed)
    elif action == 'STOP':
        stop()
        time.sleep(2)
        forward(speed)
    elif action == 'LEFT':
        turn_left(speed, TURN_DURATION)
        forward(speed)
    elif action == 'RIGHT':
        turn_right(speed, TURN_DURATION)
        forward(speed)
    elif action == 'SLOW':
        speed = speed-10
        forward(speed)
    elif action == 'DONE':
        stop()
    return speed

forward(DRIVE_SPEED)
print('Robot moving -- goal: travel straight from A to B!')
print()



# CV avoidance sequence -- hardcoded 3s segments
CV_AVOIDANCE = [
    ('RIGHT', 3.0),   # turn right to clear obstacle
    ('GO',    3.0),   # drive past obstacle
    ('LEFT',  3.0),   # turn left to resume heading
    ('GO',    3.0),   # drive to clear robot width
    ('LEFT',  3.0),   # turn left back toward original path
    ('GO',    3.0),   # rejoin path
    ('RIGHT', 3.0),   # turn right to resume original heading
]

while challenge_running:
    action = None

    # ── Check for obstacle first (higher priority than color) ──
    dist = get_distance()

    if dist < OBSTACLE_DISTANCE:
        stop()
        print(f'  Obstacle detected at {dist}cm!')

        if MODE == 'CV':
            print('[CV]  Executing hardcoded avoidance sequence...')
            execute_avoidance_sequence(CV_AVOIDANCE)
            print('[CV]  Avoidance complete -- resuming.')

        elif MODE == 'AI':
            situation = f'There is an obstacle {dist}cm directly ahead of the vehicle.'
            print(f'[AI]  Asking Phi-3 for avoidance sequence...')
            raw, sequence = ask_ai_sequence(situation)
            if sequence:
                print(f'[AI]  Phi-3 returned {len(sequence)} step sequence:')
                execute_avoidance_sequence(sequence)
                print('[AI]  Avoidance complete -- resuming.')
            else:
                stop()
                print('[AI]  ERROR: Could not parse AI response. Stopping.')
                print(f'[AI]  Raw response: {raw}')
                challenge_running = False

    else:
        # ── No obstacle -- check color card ───────────────────
        for _ in range(5):
            cap.read()
        ret, frame = cap.read()
        if not ret:
            print('ERROR: Camera read failed.')
            break

        color = detect_color(frame)

        if color:
            if MODE == 'CV':
                action_map = {
                    'red':    'STOP',
                    'green':  'GO',
                    'orange': 'GO',#'DONE',
                }
                action = action_map.get(color)
                print(f'[CV]  Color: {color:8s}  -->  Action: {action}')

            elif MODE == 'AI':
                stop()
                situation = f'The robot camera has detected the color {color}.'
                print(f'[AI]  Color: {color:8s}  -->  Asking Phi-3...')
                action = ask_ai(situation)
                if action:
                    print(f'[AI]  Phi-3 says: {action}')
                else:
                    print(f'[AI]  No valid action returned -- continuing forward.')

        # annotate and display frame
        label = f'Mode: {MODE}  |  Color: {color if color else "none"}  |  Dist: {dist:.0f}cm'
        cv2.putText(frame, label, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        feed_widget.value = bgr8_to_jpeg(frame)

    # ── Execute color action if it changed ────────────────────
    if action and action != last_action:
        execute_action(action)
        last_action = action
        if action == 'DONE':
            print()
            print('=' * 50)
            print('  MISSION COMPLETE!')
            print('=' * 50)
            challenge_running = False

    time.sleep(SCAN_INTERVAL)

  CHALLENGE STARTING -- Mode: CV



Image(value=b'', format='jpeg', height='480', width='640')

Robot moving -- goal: travel straight from A to B!

[CV]  Color: orange    -->  Action: GO
[CV]  Color: orange    -->  Action: GO
[CV]  Color: orange    -->  Action: GO
[CV]  Color: orange    -->  Action: GO
[CV]  Color: orange    -->  Action: GO
  Obstacle detected at 17.6cm!
[CV]  Executing hardcoded avoidance sequence...
  Executing: RIGHT for 3.0s
  Executing: GO for 3.0s
  Executing: LEFT for 3.0s
  Executing: GO for 3.0s
  Executing: LEFT for 3.0s
  Executing: GO for 3.0s
  Executing: RIGHT for 3.0s
[CV]  Avoidance complete -- resuming.
[CV]  Color: orange    -->  Action: GO
[CV]  Color: orange    -->  Action: GO
[CV]  Color: orange    -->  Action: GO
[CV]  Color: orange    -->  Action: GO
[CV]  Color: orange    -->  Action: GO
[CV]  Color: orange    -->  Action: GO
[CV]  Color: orange    -->  Action: GO
[CV]  Color: orange    -->  Action: GO
[CV]  Color: orange    -->  Action: GO
[CV]  Color: orange    -->  Action: GO
[CV]  Color: orange    -->  Action: GO
[CV]  Color: orange   

KeyboardInterrupt: 

---
## Stop
Run this cell at any time to stop the robot.

In [20]:
challenge_running = False
stop()
print('Robot stopped.')

Robot stopped.


---

## Debrief

Talk through these questions with your team:

1. Which mode was faster to respond -- CV or AI? Why?
2. Did you notice the robot pause in AI mode? How did that feel compared to CV mode?
3. The robot stopped moving while waiting for Phi-3. Is that the right behavior for a real car? What are the tradeoffs?
4. Did Phi-3 ever return an unexpected answer? How did the code handle it?
5. In a real self-driving car, when would you trust AI reasoning over direct sensor logic?

---

## Always clean up when you are done!

In [ ]:
stop()
cap.release()
pwm_left.stop()
pwm_right.stop()
GPIO.cleanup()
print('All cleaned up.')